In [0]:
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("table_metadata", "")
dbutils.widgets.text("catalog_name", "")

In [0]:
import json
from datetime import datetime

run_id = dbutils.widgets.get("run_id").strip()
table_metadata = dbutils.widgets.get("table_metadata")
catalog_name = dbutils.widgets.get("catalog_name").strip()

if not run_id or not table_metadata:
    raise ValueError("run_id and table_metadata are required")
if not run_id.isdigit():
    raise ValueError(
        f"run_id must be numeric (pipeline_runs.run_id is BIGINT). Got: {run_id!r}"
    )
if not catalog_name:
    raise ValueError("catalog_name is required (e.g. 'banking' — see config/*.json)")

table_metadata = json.loads(table_metadata.replace("'", '"'))

table_id = table_metadata["table_id"]
table_name = table_metadata["table_name"]

start_time = datetime.utcnow()

print("Run ID:", run_id)
print("Table ID:", table_id)
print("Table Name:", table_name)

In [0]:
%sql
create schema if not exists ${catalog_name}.gold;

In [0]:
entry_exists = spark.sql(f"""
    SELECT 1
    FROM {catalog_name}.metadata.pipeline_runs
    WHERE run_id = {run_id}
    AND table_id = {table_id}
""").count() > 0

if entry_exists:
    
    spark.sql(f"""
        UPDATE {catalog_name}.metadata.pipeline_runs
        SET
            layer = 'Gold',
            start_time = TIMESTAMP('{start_time}'),
            end_time = NULL,
            status = 'INPROGRESS',
            number_of_records = NULL,
            error_message = NULL
        WHERE run_id = {run_id}
        AND table_id = {table_id}
    """)

else:

    spark.sql(f"""
        INSERT INTO {catalog_name}.metadata.pipeline_runs
        VALUES (
            {run_id},
            {table_id},
            'Gold',
            TIMESTAMP('{start_time}'),
            NULL,
            'INPROGRESS',
            NULL,
            NULL
        )
    """)

print("Audit entry created / updated")

In [0]:
notebook_path = f"gold_transformation/{table_name}"

print("Notebook to execute:", notebook_path)

In [0]:
status = "SUCCESS"
error_message = None
records = None

try:

    result = dbutils.notebook.run(
        notebook_path,
        timeout_seconds=0
    )

    # Expect the notebook to return record count
    if result:
        records = int(result)

    print("Notebook completed successfully")
    print("Records:", records)

except Exception as e:

    status = "FAILED"
    error_message = str(e)

    print("Notebook failed")
    print(error_message)

In [0]:
end_time = datetime.utcnow()

spark.sql(f"""
    UPDATE {catalog_name}.metadata.pipeline_runs
    SET
        end_time = TIMESTAMP('{end_time}'),
        status = '{status}',
        number_of_records = {records if records else 'NULL'},
        error_message = {f"'{error_message}'" if error_message else 'NULL'}
    WHERE run_id = {run_id}
    AND table_id = {table_id}
""")

print("Audit table updated")

In [0]:
if status == "FAILED":
    raise Exception(error_message)